# RAFT · Stage 1 — Generate the synthetic dataset

Builds the RAFT training set from the fraud document corpus (AML training domain, other
domains as distractors). Each example = a question + the **golden** document + several
**distractor** documents + a chain-of-thought answer that cites the exact typology and rule.

**Runtime:** ~5 min (sample) to hours (large). **Cost:** a few dollars of teacher tokens —
see `README.md`; the hosting fee in Stage 3 is the figure that matters.

Runs interactively **and** non-interactively via papermill: `papermill 1_gen.ipynb out.ipynb -f parameters/gpt-4.1-mini.yaml`.
Model names are parameters, never literals. Re-verify catalogue + region quota before running.

In [ ]:
# papermill parameters — overridden by parameters/*.yaml. Never hard-code model names below.
region = "swedencentral"
teacher_model = "gpt-4.1"
student_model = "gpt-4.1-mini"
seed_source = "synthetic"  # synthetic | traces
seed_questions_path = "data/seed_questions.jsonl"
corpus_manifest = "../../fabric/lakehouse/corpus/manifest.yaml"
training_domain = "aml"
n_questions = 60
n_distractors = 4
oracle_probability = 0.8
train_path = "data/raft_train.jsonl"
val_path = "data/raft_val.jsonl"
val_fraction = 0.15
seed = 42

In [ ]:
import json, os, random, time, pathlib
import yaml

# Config comes from the environment, never from hard-coded endpoints/keys.
FOUNDRY_ENDPOINT = os.environ.get("AI_FOUNDRY_ENDPOINT", "")
HERE = pathlib.Path.cwd()
CORPUS_ROOT = (HERE / corpus_manifest).resolve().parent
random.seed(seed)
t0 = time.time()
print(f"Stage 1 · teacher={teacher_model} · region={region} · domain={training_domain}")
print(f"Corpus: {CORPUS_ROOT}")
print("Estimated cost: a few USD of teacher tokens (see README hosting warning for Stage 3).")

In [ ]:
# Canonical system message — MUST be identical between training and inference (raft.instructions.md).
SYSTEM_MESSAGE = (
    "You are an AML analyst assistant. Answer only from the provided documents. "
    "Cite the exact typology and rule. If the documents do not support an answer, say so. "
    "Output is advisory; a human must approve any filing."
)

manifest = yaml.safe_load((CORPUS_ROOT / "manifest.yaml").read_text(encoding="utf-8"))
assert manifest["training_domain"] == training_domain, "manifest training_domain must match parameter"
docs = manifest["documents"]
golden = [d for d in docs if d["domain"] == training_domain]
distractors = [d for d in docs if d["domain"] != training_domain]
print(f"{len(golden)} golden (aml) docs, {len(distractors)} distractor docs")

def read_doc(d):
    return (CORPUS_ROOT / d["path"]).read_text(encoding="utf-8")

In [ ]:
# Seed questions: synthetic (from the corpus) or real analyst traces (WS-5 export).
if seed_source == "traces":
    questions = [json.loads(l) for l in pathlib.Path(seed_questions_path).read_text(encoding="utf-8").splitlines() if l.strip()]
else:
    # Synthesize questions grounded on each golden doc's typology.
    questions = []
    for i in range(n_questions):
        g = golden[i % len(golden)]
        questions.append({"id": f"Q-{i+1:04d}", "domain": training_domain, "golden": g["path"],
                          "typology": g.get("typology", ""),
                          "question": f"Explain and classify a case matching '{g['title']}' and cite the exact rule."})
print(f"{len(questions)} seed questions ({seed_source})")

In [ ]:
# Teacher client (Azure OpenAI on the Foundry resource). Uses AAD, no keys in the notebook.
def teacher_answer(question, golden_text, context_texts):
    """Return a chain-of-thought answer grounded in the golden document."""
    try:
        from azure.identity import DefaultAzureCredential, get_bearer_token_provider
        from openai import AzureOpenAI
        token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default")
        client = AzureOpenAI(azure_endpoint=FOUNDRY_ENDPOINT, azure_ad_token_provider=token_provider, api_version="2024-10-21")
        ctx = "\n\n".join(context_texts)
        msg = [{"role": "system", "content": SYSTEM_MESSAGE},
               {"role": "user", "content": f"{ctx}\n\nQuestion: {question}\nAnswer with '##Reason:' then '##Answer:'."}]
        r = client.chat.completions.create(model=teacher_model, messages=msg, temperature=0.2)
        return r.choices[0].message.content
    except Exception as e:  # offline / no endpoint — deterministic placeholder so the notebook runs dry.
        return f"##Reason: (offline placeholder — teacher unavailable: {type(e).__name__}). ##Answer: cite the golden document's typology and rule."

In [ ]:
def build_example(q):
    g = next((d for d in golden if d["path"] == q.get("golden")), random.choice(golden))
    golden_text = f"<DOCUMENT id=\"{g['path']}\">\n{read_doc(g)}\n</DOCUMENT>"
    picks = random.sample(distractors, min(n_distractors, len(distractors)))
    distractor_texts = [f"<DOCUMENT id=\"{d['path']}\">\n{read_doc(d)}\n</DOCUMENT>" for d in picks]
    # Oracle probability: sometimes omit the golden doc so the model learns to abstain.
    include_golden = random.random() < oracle_probability
    context = ([golden_text] if include_golden else []) + distractor_texts
    random.shuffle(context)
    answer = teacher_answer(q["question"], golden_text, context)
    user = "\n".join(context) + f"\n\nQuestion: {q['question']}"
    return {"messages": [{"role": "system", "content": SYSTEM_MESSAGE},
                          {"role": "user", "content": user},
                          {"role": "assistant", "content": answer}]}

examples = [build_example(q) for q in questions]
random.shuffle(examples)
n_val = max(1, int(len(examples) * val_fraction))
val, train = examples[:n_val], examples[n_val:]
print(f"{len(train)} train / {len(val)} val examples")

In [ ]:
# Foundry training files: JSONL, Chat Completions format, UTF-8 WITH BOM (utf-8-sig), < 512 MB.
def write_jsonl_bom(path, rows):
    p = pathlib.Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8-sig") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"wrote {len(rows)} -> {p}")

write_jsonl_bom(train_path, train)
write_jsonl_bom(val_path, val)
print(f"Stage 1 done in {time.time()-t0:.0f}s. Next: 2_finetune.ipynb")